# 04 · Analysis: leaflet Tables 3–5

Evaluate a simulated, noisy coupled scan the way LD Physics Leaflet P6.3.3.1 does: mark the entire
width of each line, take its peak centre ("Calculate Peak Center" in the LD software), get λ from
Bragg's law per order, and average the orders. Everything lives in `braggsim.analysis`; this
notebook only runs it and shows the result next to the leaflet's numbers.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from braggsim.analysis import (
    PEAK_HALF_WINDOW_DEG,
    find_line_peaks,
    order_means_pm,
    wavelength_table,
)
from braggsim.constants import MO_KA_PM, MO_KB_PM
from braggsim.detector import sample_counts
from braggsim.scan import coupled_betas, expected_rate

plt.rcParams.update(
    {
        "axes.grid": True,
        "grid.color": "#e5e5e5",
        "lines.linewidth": 1.2,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
DATA, MODEL, ALT = "#1a1a19", "#2a78d6", "#eb6834"

## A simulated measurement

Leaflet settings: U = 35 kV, I = 1 mA, 2° → 25° in Δβ = 0.1° steps, Δt = 10 s per step. The
expected rate R̄(β) comes from the calibrated model; Poisson noise is the only random step.

In [ ]:
DT_S = 10.0
beta = coupled_betas(2.0, 25.0, 0.1)
r_bar = expected_rate(beta, 35.0, 1.0)
rate = sample_counts(r_bar, DT_S, np.random.default_rng(2026)) / DT_S

rows = wavelength_table(beta, rate, DT_S)
table = pd.DataFrame(rows).set_index(["line", "n"]).sort_index()
table.round(3)

In [ ]:
# Leaflet Tables 3 and 4 (measured) for comparison, LD P6.3.3.1.
leaflet = pd.DataFrame(
    [("Ka", 1, 7.24, 71.08), ("Ka", 2, 14.60, 71.09), ("Ka", 3, 22.20, 71.04),
     ("Kb", 1, 6.42, 63.07), ("Kb", 2, 12.94, 63.15), ("Kb", 3, 19.58, 63.01)],
    columns=["line", "n", "theta_deg", "lambda_pm"],
).set_index(["line", "n"])  # fmt: skip
table[["theta_deg", "theta_err_deg", "lambda_pm", "lambda_err_pm"]].join(
    leaflet, rsuffix="_leaflet"
).round(3)

## Table 5: mean over the orders

In [ ]:
means = order_means_pm(rows)
pd.DataFrame(
    {
        "simulated (pm)": {k: f"{m:.3f} ± {e:.3f}" for k, (m, e) in means.items()},
        "leaflet mean (pm)": {"Ka": 71.07, "Kb": 63.08},
        "literature (pm)": {"Ka": MO_KA_PM, "Kb": MO_KB_PM},
    }
)

## The marked peaks

Grey bands are the marked regions (±`PEAK_HALF_WINDOW_DEG` = 0.7°, cut halfway to the neighbouring peak); the dashed
lines are the peak centres. Linear and log scale.

In [ ]:
peaks = find_line_peaks(beta, rate, DT_S)
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
for ax, log in zip(axes, (False, True), strict=True):
    ax.plot(beta, rate, ".", color=DATA, ms=3, label="simulated counts / Δt")
    ax.plot(beta, r_bar, color=MODEL, label="expected R̄(β)")
    mids = (peaks[1:] + peaks[:-1]) / 2
    for p, lo, hi in zip(peaks, np.r_[-np.inf, mids], np.r_[mids, np.inf], strict=True):
        ax.axvspan(
            max(p - PEAK_HALF_WINDOW_DEG, lo),
            min(p + PEAK_HALF_WINDOW_DEG, hi),
            color="#000",
            alpha=0.05,
        )
    for r in rows:
        ax.axvline(r["theta_deg"], color=ALT, ls="--", lw=0.8)
    ax.set_yscale("log" if log else "linear")
    ax.set_ylabel("R (1/s)")
axes[0].legend()
axes[1].set_xlabel("β (°)")
fig.tight_layout()

## Why a centroid and not a Gaussian fit

Each line is a blend: Kα1 + Kα2 and Kβ1,3 + Kβ2, and the leaflet's Table 1 wavelengths are their
intensity-weighted means. Kβ2 (62.10 pm) lies 1.15 pm below Kβ1,3 (63.25 pm): 0.12° apart in
1st order (unresolved), 0.37° in 3rd order (a separate shoulder). A single Gaussian fitted to the
top of the peak locks onto Kβ1,3 in the higher orders and gives λ(Kβ) ≈ 63.2 pm. The centroid of
the whole marked peak returns the blend mean, as the leaflet's workflow does. Kα1/Kα2 stay
blended through 3rd order at these slits (0.14° apart, σ ≈ 0.12°).

In [ ]:
pd.DataFrame(
    {
        hw: {k: m for k, (m, e) in order_means_pm(wavelength_table(beta, r_bar, DT_S, hw)).items()}
        for hw in (0.4, 0.5, 0.6, 0.7, 0.8)
    }
).rename_axis("half-window (°) →").round(3)  # noise-free: narrow windows miss Kβ2